# 第18章 回归

本 notebook 用一个小型光度红移教学例子说明回归流程：读取数据、切分训练/测试集、建立线性回归 baseline、比较简单非线性扩展。

## 学习目标

- 理解回归任务与分类任务的区别。
- 手写一个最小线性回归 baseline。
- 用 MAE 和 RMSE 评估测试误差。
- 理解多项式特征为什么可能改善拟合，也可能带来过拟合。

In [ ]:
import csv
import math
from pathlib import Path

data_path = Path("../../data/small/photometric_redshift_demo.csv")
with data_path.open(newline="", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

for row in rows:
    for key in ("u_g", "g_r", "r_i", "i_z", "specz"):
        row[key] = float(row[key])

print("loaded rows:", len(rows))
print("redshift range:", rows[0]["specz"], "to", rows[-1]["specz"])


## 一个固定的教学切分

为了保持 notebook 结果稳定，我们使用固定索引划分测试集。真实项目通常需要更系统的交叉验证或更大的样本。

In [ ]:
test_indices = {2, 6, 10}
train_rows = [row for index, row in enumerate(rows) if index not in test_indices]
test_rows = [row for index, row in enumerate(rows) if index in test_indices]

print("train:", len(train_rows), "test:", len(test_rows))
print("test galaxy ids:", [row["galaxy_id"] for row in test_rows])


## 最小线性回归 baseline

这里先只使用一个颜色特征 `g-r`。这不是最佳科学选择，而是一个方便理解的一维教学起点。

In [ ]:
def fit_line(xs, ys):
    x_mean = sum(xs) / len(xs)
    y_mean = sum(ys) / len(ys)
    numerator = sum((x - x_mean) * (y - y_mean) for x, y in zip(xs, ys))
    denominator = sum((x - x_mean) ** 2 for x in xs)
    slope = numerator / denominator
    intercept = y_mean - slope * x_mean
    return slope, intercept


def predict_line(x_value, slope, intercept):
    return slope * x_value + intercept


x_train = [row["g_r"] for row in train_rows]
y_train = [row["specz"] for row in train_rows]
x_test = [row["g_r"] for row in test_rows]
y_test = [row["specz"] for row in test_rows]

slope, intercept = fit_line(x_train, y_train)
line_predictions = [predict_line(x, slope, intercept) for x in x_test]

print("slope =", round(slope, 3))
print("intercept =", round(intercept, 3))
print("test predictions =", [round(value, 3) for value in line_predictions])


## 常见回归指标

MAE 反映平均绝对偏差，RMSE 对大误差更敏感。两者都值得看。

In [ ]:
def mean_absolute_error(y_true, y_pred):
    return sum(abs(a - b) for a, b in zip(y_true, y_pred)) / len(y_true)


def root_mean_squared_error(y_true, y_pred):
    mse = sum((a - b) ** 2 for a, b in zip(y_true, y_pred)) / len(y_true)
    return math.sqrt(mse)


line_mae = mean_absolute_error(y_test, line_predictions)
line_rmse = root_mean_squared_error(y_test, line_predictions)
print("linear mae =", round(line_mae, 4))
print("linear rmse =", round(line_rmse, 4))


## 一个手写的二次多项式回归

为了保持 notebook 在最小环境下也能运行，我们用高斯消元手写一个小型二次拟合。重点是理解“多项式特征”这个思想。

In [ ]:
def solve_linear_system(matrix, values):
    augmented = [row[:] + [value] for row, value in zip(matrix, values)]
    size = len(augmented)

    for pivot in range(size):
        pivot_row = max(range(pivot, size), key=lambda idx: abs(augmented[idx][pivot]))
        augmented[pivot], augmented[pivot_row] = augmented[pivot_row], augmented[pivot]

        pivot_value = augmented[pivot][pivot]
        for column in range(pivot, size + 1):
            augmented[pivot][column] /= pivot_value

        for row_index in range(size):
            if row_index == pivot:
                continue
            factor = augmented[row_index][pivot]
            for column in range(pivot, size + 1):
                augmented[row_index][column] -= factor * augmented[pivot][column]

    return [row[-1] for row in augmented]


def fit_quadratic(xs, ys):
    design = [[1.0, x, x * x] for x in xs]
    matrix = [[0.0] * 3 for _ in range(3)]
    values = [0.0] * 3

    for row, y_value in zip(design, ys):
        for i in range(3):
            values[i] += row[i] * y_value
            for j in range(3):
                matrix[i][j] += row[i] * row[j]

    return solve_linear_system(matrix, values)


def predict_quadratic(x_value, coefficients):
    a0, a1, a2 = coefficients
    return a0 + a1 * x_value + a2 * x_value * x_value


quadratic_coeffs = fit_quadratic(x_train, y_train)
quadratic_predictions = [predict_quadratic(x, quadratic_coeffs) for x in x_test]
quadratic_mae = mean_absolute_error(y_test, quadratic_predictions)
quadratic_rmse = root_mean_squared_error(y_test, quadratic_predictions)

print("quadratic coefficients =", [round(value, 3) for value in quadratic_coeffs])
print("quadratic mae =", round(quadratic_mae, 4))
print("quadratic rmse =", round(quadratic_rmse, 4))


In [ ]:
comparison_rows = []
for row, line_pred, quad_pred in zip(test_rows, line_predictions, quadratic_predictions):
    comparison_rows.append({
        "galaxy_id": row["galaxy_id"],
        "truth": round(row["specz"], 3),
        "linear": round(line_pred, 3),
        "quadratic": round(quad_pred, 3),
    })

comparison_rows


## 如果安装了 scikit-learn

下面这个单元是可选的。它把最小教学实现与常见生态中的做法连起来。

In [ ]:
try:
    from sklearn.ensemble import RandomForestRegressor
    from sklearn.linear_model import Ridge
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import PolynomialFeatures
except ModuleNotFoundError:
    print("scikit-learn 未安装；已跳过标准库回归示例。")
else:
    x_train_multi = [[row["g_r"], row["r_i"]] for row in train_rows]
    x_test_multi = [[row["g_r"], row["r_i"]] for row in test_rows]

    ridge_model = make_pipeline(PolynomialFeatures(degree=2, include_bias=False), Ridge(alpha=0.01))
    ridge_model.fit(x_train_multi, y_train)
    ridge_predictions = ridge_model.predict(x_test_multi)
    print("ridge mae =", round(mean_absolute_error(y_test, ridge_predictions), 4))

    forest_model = RandomForestRegressor(n_estimators=50, random_state=42)
    forest_model.fit(x_train_multi, y_train)
    forest_predictions = forest_model.predict(x_test_multi)
    print("random-forest mae =", round(mean_absolute_error(y_test, forest_predictions), 4))


## 小结

这个例子说明了一个很重要的教学节奏：先用最小 baseline 理解回归流程，再逐步引入更灵活的特征和更成熟的工具。真正重要的不是模型名字，而是误差是否可信、样本覆盖范围是否合理。